# Bell Pepper Dataset generator

In [6]:
!pip uninstall -y transformers huggingface_hub
!pip install --upgrade transformers huggingface_hub

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [7]:
# imports

import os
from IPython.display import display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import gradio as gr
import torch

from styles import CSS

In [8]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
# del model, inputs, tokenizer, outputs
# gc.collect()
# torch.cuda.empty_cache()

In [10]:
system_message = """
You are a helpful data collector for a bell pepper farmer.
Generate a dataset for a bell pepper farm. For each bell pepper, you need to generate the following data:
- Seed variant
- Disease resistance
- Fertilizer used
- Harvesting dates
- Pruning dates
- Pest control dates
and any other relevant data for the bell pepper farm.

then return the data in csv format.
"""

user_prompt = f"""
I need you to generate a dataset for a bell pepper farm. I need it to make for future reference as well as for training a model.
"""


In [11]:
MODEL = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

In [20]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(quant=True, max_new_tokens=80):
  messages = [{"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]
  
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)
  streamer = TextStreamer(tokenizer)
  
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

  return tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

  # return tokenizer.decode(outputs[0])

In [21]:
def chat(history):
    response = generate()
    
    return response

In [22]:
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Bell Pepper Farm Data Generator") as ui:

    chatbox = gr.Chatbot(type="messages", value=[])

    generate_btn = gr.Button("Generate")

    generate_btn.click(
        fn=chat,
        inputs=[chatbox], # Pass the current chat history in
        outputs=chatbox
    )


ui.launch()



ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/uvicorn/protocols/http/h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/fastapi/applications.py", line 1138, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/midd

Running on local URL:  http://127.0.0.1:7863


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/uvicorn/protocols/http/h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/fastapi/applications.py", line 1138, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/Users/seunaminu/Library/Python/3.9/lib/python/site-packages/starlette/midd

ValueError: When localhost is not accessible, a shareable link must be created. Please set share=True or check your proxy settings to allow access to localhost.